<a href="https://colab.research.google.com/github/aaronab810/Dubai-Real-Estate-NLP/blob/main/notebooks/Topic_discovery_for_SMDI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Format

the first part of this notebook is for initial bertopic discorvery from all of social media real estate posts/comments.

## **"BERTOPIC Discovery" Section**
bertopic discovery



## **"START FROM HERE" Section**
this section is the clustering experimentation for reduction of bertopic topics.

## AutoModerator correction (6 September 2026)

The training path excludes the known Reddit AutoModerator account before encoding.
The resume path excludes the same rows from both the saved dataframe and its
embedding matrix, recounts topics, and saves an exclusion audit. Resume cleaning
does not undo the effect bots had on the fitted BERTopic model: rerun discovery
for a new model. Existing outputs have been cleared because their counts are stale.
Topic -1 stays unassigned and does not enter semantic clustering. Group numbers
can change, so the old five-group SMDI mapping is retained only as a historical
example; fill REVIEWED_SMDI_MAPPING after inspecting the new groups.

Before the clean-corpus refit below, discovery loaded master_processed.csv
and appended Reddit records, while validation used
master_processed_analysis_clean.parquet. Those historical corpora are not
interchangeable; the AutoModerator-only correction preserved that earlier input.

## Clean-corpus refit (7 September 2026)

Discovery now defaults to `final_pipeline_outputs/master_processed_analysis_clean.parquet`
(CSV fallback). It does not append raw Reddit exports in this mode. The historical
append path remains available through an explicit flag for reproducibility.
Known AutoModerator accounts are excluded before training. New artifacts use a
versioned output directory, so old models and group mappings are not overwritten.
The local executed refit's manifest is in `analysis/refit_2026_09_07/refit_manifest.json`.
That local run saves assigned-cluster strengths rather than a full probability
matrix; this notebook retains the existing full-probability option.


# BERTOPIC Discovery

Load the final cleaned corpus used for validation. Additional raw Reddit exports are only loaded when LEGACY_APPEND_REDDIT is explicitly enabled for a historical comparison.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)

try:
    from IPython.display import display
except Exception:
    display = print


In [ ]:
# Final cleaned corpus is the default, matching validation provenance.
BASE = "/content/drive/MyDrive/Dubai_Real_Estate_Data"
LEGACY_APPEND_REDDIT = False
RUN_NAME = "legacy_refit_2026_09_07" if LEGACY_APPEND_REDDIT else "clean_refit_2026_09_07"
stem = "master_processed" if LEGACY_APPEND_REDDIT else "master_processed_analysis_clean"
SOCIAL_PARQUET = f"{BASE}/final_pipeline_outputs/{stem}.parquet"
SOCIAL_CSV = f"{BASE}/final_pipeline_outputs/{stem}.csv"
SAVE_DIR = f"{BASE}/SMDI creation/{RUN_NAME}"
OUT_DIR = f"{SAVE_DIR}/dld_social_correlation"
DLD_RAW_PATH = f"{BASE}/dld/transactions-2026-06-19.csv"
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(SOCIAL_PARQUET) and not os.path.exists(SOCIAL_CSV):
    raise FileNotFoundError(f"Run the merge notebook's final export first: {SOCIAL_PARQUET}")
print("Model output directory:", SAVE_DIR)


In [ ]:
if not LEGACY_APPEND_REDDIT:
    social = (pd.read_parquet(SOCIAL_PARQUET) if os.path.exists(SOCIAL_PARQUET)
              else pd.read_csv(SOCIAL_CSV, low_memory=False))
    if social.duplicated(["platform", "id"]).any():
        raise ValueError("Duplicate platform + record IDs in final cleaned corpus")
    print("Loaded final cleaned corpus:", len(social), "records")
else:
    reddit_posts = pd.read_csv(f"{BASE}/Prajwal final reddit/combined_reddit_posts.csv", low_memory=False)
    reddit_comments = pd.read_csv(f"{BASE}/Prajwal final reddit/combined_reddit_comments.csv", low_memory=False)

    social = pd.read_csv(SOCIAL_CSV, low_memory=False)
    print("Loaded social CSV:", SOCIAL_CSV)

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_posts_social = pd.DataFrame({
        "id": "reddit_post_" + reddit_posts["post_id"].astype(str),

        "raw_text": reddit_posts["post_text"],
        "clean_text": reddit_posts["post_text"],

        "date": pd.to_datetime(reddit_posts["date"]),

        "platform": "reddit",
        "source_label": "reddit",

        "post_type": "reddit_post",

        "platform_interaction_value": reddit_posts["post_score"],

        "platform_interaction_definition":
            "Reddit source-native score for post",

        "likeCount": reddit_posts["post_score"],

        "retweetCount": np.nan,

        "replyCount": reddit_posts["num_comments"],

        "quoteCount": np.nan,

        "is_retweet": False,
        "is_reply": False,

        "username": reddit_posts["post_author"],

        "url": reddit_posts["url"],

        "subreddit": reddit_posts["subreddit"],

        "parent_id": np.nan,

        "link_id": reddit_posts["post_id"],

        "thread_id": reddit_posts["post_id"]
    })

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_comments_social = pd.DataFrame({
        "id": "reddit_comment_" + reddit_comments["comment_id"].astype(str),

        "raw_text": reddit_comments["body"],
        "clean_text": reddit_comments["body"],

        "date": pd.to_datetime(reddit_comments["date"]),

        "platform": "reddit",
        "source_label": "reddit",

        "post_type": "reddit_comment",

        "platform_interaction_value": reddit_comments["comment_score"],

        "platform_interaction_definition":
            "Reddit source-native score for comment",

        "likeCount": reddit_comments["comment_score"],

        "retweetCount": np.nan,

        "replyCount": np.nan,

        "quoteCount": np.nan,

        "is_retweet": False,
        "is_reply": True,

        "username": reddit_comments["comment_author"],

        "url": "https://www.reddit.com" + reddit_comments["permalink"],

        "subreddit": np.nan,

        "parent_id": reddit_comments["parent_id"],

        "link_id": reddit_comments["post_id"],

        "thread_id": reddit_comments["post_id"]
    })

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_comments.head()

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_comments_social.head()

In [ ]:
if LEGACY_APPEND_REDDIT:
    for col in social.columns:
        if col not in reddit_posts_social.columns:
            reddit_posts_social[col] = np.nan

    for col in social.columns:
        if col not in reddit_comments_social.columns:
            reddit_comments_social[col] = np.nan

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_posts_social = reddit_posts_social[social.columns]
    reddit_comments_social = reddit_comments_social[social.columns]

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_new = pd.concat(
        [reddit_posts_social, reddit_comments_social],
        ignore_index=True
    )

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_new.head()

In [ ]:
if LEGACY_APPEND_REDDIT:
    # Only existing Reddit IDs in the social dataset
    existing_reddit_ids = set(
        social.loc[social["platform"] == "reddit", "id"]
    )

    # Check whether each new Reddit record already exists
    reddit_new["exists"] = reddit_new["id"].isin(existing_reddit_ids)

    reddit_new["exists"].value_counts()

In [ ]:
if LEGACY_APPEND_REDDIT:

    print(len(reddit_posts) + len(reddit_comments))

In [ ]:
social["platform"].value_counts()

In [ ]:
len(social)

In [ ]:
if LEGACY_APPEND_REDDIT:
    reddit_to_add = reddit_new.loc[~reddit_new["exists"]].copy()

    print(f"Rows to add: {len(reddit_to_add)}")
    reddit_to_add = reddit_to_add.drop(columns="exists")

In [ ]:
if LEGACY_APPEND_REDDIT:
    social = pd.concat(
        [social, reddit_to_add],
        ignore_index=True
    )

In [ ]:
print("Total rows:", len(social))
print()
print(social["platform"].value_counts())

In [ ]:
len(social)

In [ ]:
social["id"].duplicated().sum()

In [ ]:
if LEGACY_APPEND_REDDIT:
    # Reddit IDs already in social
    existing_reddit_ids = set(social.loc[social["platform"] == "reddit", "id"])

    # IDs from the latest Reddit scrape
    new_reddit_ids = set(reddit_new["id"])

    # IDs present in social but missing from the new scrape
    missing_from_scrape = existing_reddit_ids - new_reddit_ids

    print(f"Missing IDs: {len(missing_from_scrape)}")
    print(missing_from_scrape)

In [ ]:
len(social)

Topic discovery

In [ ]:
!pip install bertopic

In [ ]:
import os
import pandas as pd

def known_automoderator_mask(frame):
    """Only the known Reddit moderation account; never a generic 'bot' substring."""
    required = {"platform", "username"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"AutoModerator filtering requires columns: {sorted(missing)}")
    platform = frame["platform"].fillna("").astype(str).str.strip().str.casefold()
    author = (frame["username"].fillna("").astype(str).str.strip().str.casefold()
              .str.replace(r"^/?u/", "", regex=True))
    return platform.eq("reddit") & author.eq("automoderator")

# Keep an auditable copy of every excluded bot record before text cleanup.
bot_mask = known_automoderator_mask(social)
bot_audit_dir = f"{SAVE_DIR}/quality_audit"
os.makedirs(bot_audit_dir, exist_ok=True)
social.loc[bot_mask].assign(exclusion_reason="known_reddit_automoderator").to_csv(
    f"{bot_audit_dir}/automoderator_excluded_training.csv", index=False)
print(f"Known AutoModerator records excluded before embeddings: {int(bot_mask.sum()):,}")
social = social.loc[~bot_mask].copy()
social = social.dropna(subset=["clean_text"]).copy()
social = social.loc[social["clean_text"].str.strip().ne("")].reset_index(drop=True)
assert not known_automoderator_mask(social).any()
print(f"Records entering BERTopic: {len(social):,}")


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    social["clean_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)


In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=None,      # Use OUR embeddings
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

In [ ]:
topics, probs = topic_model.fit_transform(
    social["clean_text"].tolist(),
    embeddings
)

social["topic"] = topics

In [ ]:
topic_info = topic_model.get_topic_info()

topic_info

In [ ]:
topic_model.visualize_topics()

In [ ]:
len(social)

now onto topic merging and getting fewer topics and creating the SMDIs

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(
    social["clean_text"].tolist()
)

In [ ]:
topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)

In [ ]:
topic_info = topic_model.get_topic_info()

topic_info.to_csv(
    "bertopic_topics.csv",
    index=False
)

In [ ]:
# SAVE_DIR comes from the versioned configuration above.
import numpy as np
import pickle
import os
os.makedirs(SAVE_DIR, exist_ok=True)

# Embeddings
np.save(f"{SAVE_DIR}/embeddings.npy", embeddings)

# BERTopic model
topic_model.save(f"{SAVE_DIR}/bertopic_model")

# Data
social.to_csv(f"{SAVE_DIR}/social_with_topics.csv", index=False)

social.to_pickle(f"{SAVE_DIR}/social_with_topics.pkl")

# Topic info
topic_info.to_csv(f"{SAVE_DIR}/topic_info.csv", index=False)

# Hierarchy
with open(f"{SAVE_DIR}/hierarchical_topics.pkl", "wb") as f:
    pickle.dump(hierarchical_topics, f)

print("Everything saved successfully!")

#START FROM HERE
the above code was the models which have beem exported. importing done below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Default to the cleaned-corpus refit. Choose a different version explicitly.
SAVE_DIR = "/content/drive/MyDrive/Dubai_Real_Estate_Data/SMDI creation/clean_refit_2026_09_07"


In [ ]:
import os

print(os.listdir(SAVE_DIR))

In [ ]:
import os
import numpy as np
import pandas as pd

def known_automoderator_mask(frame):
    """Only the known Reddit moderation account; never a generic 'bot' substring."""
    required = {"platform", "username"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"AutoModerator filtering requires columns: {sorted(missing)}")
    platform = frame["platform"].fillna("").astype(str).str.strip().str.casefold()
    author = (frame["username"].fillna("").astype(str).str.strip().str.casefold()
              .str.replace(r"^/?u/", "", regex=True))
    return platform.eq("reddit") & author.eq("automoderator")

# Read the saved dataframe and embeddings together; their row order must match.
embeddings = np.load(f"{SAVE_DIR}/embeddings.npy")
social = (pd.read_parquet(f"{SAVE_DIR}/social_with_topics.parquet")
          if os.path.exists(f"{SAVE_DIR}/social_with_topics.parquet")
          else pd.read_pickle(f"{SAVE_DIR}/social_with_topics.pkl"))
topic_info = pd.read_csv(f"{SAVE_DIR}/topic_info.csv")
if embeddings.ndim != 2 or len(embeddings) != len(social):
    raise ValueError("Saved embeddings and social rows do not align. Regenerate them together.")
bot_mask = known_automoderator_mask(social)
os.makedirs(f"{SAVE_DIR}/quality_audit", exist_ok=True)
social.loc[bot_mask].assign(exclusion_reason="known_reddit_automoderator").to_csv(
    f"{SAVE_DIR}/quality_audit/automoderator_excluded_resume.csv", index=False)
keep = ~bot_mask.to_numpy()
embeddings = embeddings[keep]
social = social.loc[~bot_mask].reset_index(drop=True)
print(f"Known AutoModerator records excluded on resume: {int(bot_mask.sum()):,}")
assert len(social) == len(embeddings)
assert not known_automoderator_mask(social).any()

# Recount surviving assignments; cached Count values include the excluded bots.
counts = social["topic"].value_counts()
topic_info = topic_info.loc[topic_info["Topic"].isin(counts.index)].copy()
topic_info["Count"] = topic_info["Topic"].map(counts).astype(int)
assert topic_info["Count"].sum() == len(social)
print("Resume preserves this saved run's topic IDs. Removing rows here does not refit its model.")


In [ ]:
# Use the already aligned, filtered embeddings from the preceding cell.
# Reloading embeddings.npy here would reintroduce the excluded rows.
assert len(embeddings) == len(social)
print("Embedding shape:", embeddings.shape)
print("Number of records:", len(social))
print("Number of topic entries, including outlier -1 if present:", len(topic_info))


Topic grouping

In [ ]:
display(social)

In [ ]:
social[["clean_text", "topic"]].head(20)

In [ ]:
social["topic"].value_counts().sort_index()

In [ ]:
topic_embeddings = social["topic"]

print(topic_embeddings.shape)

In [ ]:
print(social.columns.tolist())
print(topic_info.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering

# --------------------------------------------------
# 1. Load the post embeddings
# --------------------------------------------------
# Reuse the filtered matrix; do not reload the unfiltered saved cache.
assert len(embeddings) == len(social)
assert not known_automoderator_mask(social).any()

print("Post embeddings:", embeddings.shape)
print("Posts:", len(social))
print("Topics:", len(topic_info))


# --------------------------------------------------
# 2. Create one embedding for each topic
#    by averaging the embeddings of its posts
# --------------------------------------------------
topic_embeddings = []
valid_topics = []

for topic_id in topic_info.loc[topic_info["Topic"].ne(-1), "Topic"]:

    # Posts belonging to this topic
    mask = social["topic"].values == topic_id

    # Their embeddings
    topic_emb = embeddings[mask]

    # Only keep topics that have posts
    if len(topic_emb) > 0:
        topic_embeddings.append(topic_emb.mean(axis=0))
        valid_topics.append(topic_id)

# Convert to numpy array
if not topic_embeddings:
    raise ValueError("No non-outlier topics remain for semantic grouping.")
topic_embeddings = np.vstack(topic_embeddings)

print("Topic embeddings:", topic_embeddings.shape)

In [ ]:
# --------------------------------------------------
# 3. Cluster the topics into 25 semantic groups
# --------------------------------------------------
if len(valid_topics) < 25:
    raise ValueError("Fewer than 25 non-outlier topics remain. Review the group count before clustering.")
cluster_model = AgglomerativeClustering(
    n_clusters=25,
    metric="cosine",
    linkage="average"
)

groups = cluster_model.fit_predict(topic_embeddings)

# Any new grouping invalidates a previously reviewed numeric mapping.
REVIEWED_SMDI_MAPPING = {}


# --------------------------------------------------
# 4. Add semantic group to topic_info
# --------------------------------------------------
topic_info = topic_info[
    topic_info["Topic"].isin(valid_topics)
].copy()

topic_info["semantic_group"] = topic_info["Topic"].map(dict(zip(valid_topics, groups)))
assert -1 not in valid_topics

print("Semantic groups created:", topic_info["semantic_group"].nunique())

display(
    topic_info[
        ["Topic", "Count", "Name", "semantic_group"]
    ].sort_values(["semantic_group", "Count"], ascending=[True, False])
)

In [ ]:
# Summary of the 25 semantic groups
group_summary = (
    topic_info
    .groupby("semantic_group")
    .agg(
        Number_of_Topics=("Topic", "count"),
        Number_of_Posts=("Count", "sum")
    )
    .reset_index()
    .sort_values("semantic_group")
)

display(group_summary)

In [ ]:
group_summary.columns = [
    "Semantic Group",
    "Number of Topics",
    "Number of Posts"
]

display(group_summary)

In [ ]:
# Change this number to inspect any group from 0 to 24
group_number = 16

group_topics = (
    topic_info[topic_info["semantic_group"] == group_number]
    [["Topic", "Count", "Name", "Representation"]]
    .sort_values("Count", ascending=False)
)

print(f"Semantic Group {group_number}")
print(f"Number of topics: {len(group_topics)}")
print(f"Number of posts: {group_topics['Count'].sum():,}")

display(group_topics)

In [ ]:
import matplotlib.pyplot as plt

# Make sure groups are in order 0–24
plot_data = group_summary.sort_values("Semantic Group")

plt.figure(figsize=(14, 8))

plt.bar(
    plot_data["Semantic Group"].astype(str),
    plot_data["Number of Posts"]
)

plt.xlabel("Semantic Group")
plt.ylabel("Number of Posts")
plt.title("Number of Posts by Semantic Group")

plt.xticks(range(25))
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# SMDI POST FREQUENCY
# ============================================================

# Make a copy so we don't accidentally alter your original data
smdi_df = social.copy()

# ------------------------------------------------------------
# 1. CHECK YOUR COLUMN NAMES
# ------------------------------------------------------------

print("Columns in dataset:")
print(smdi_df.columns.tolist())

In [ ]:
# ============================================================
# 1. CREATE TOPIC → SEMANTIC GROUP MAPPING
# ============================================================

topic_to_group = (
    topic_info
    [['Topic', 'semantic_group']]
    .drop_duplicates()
    .set_index('Topic')['semantic_group']
    .to_dict()
)

print("Topic → Semantic Group mapping created.")
print("Number of topics mapped:", len(topic_to_group))

# Show first few mappings
print("\nExample mappings:")
for topic, group in list(topic_to_group.items())[:10]:
    print(f"Topic {topic} → Group {group}")

In [ ]:
# ============================================================
# 2. ASSIGN SEMANTIC GROUP TO EVERY POST
# ============================================================

social['semantic_group'] = social['topic'].map(topic_to_group)

# Check result
print("Posts with semantic group:")
display(
    social[
        ['id', 'date', 'topic', 'semantic_group']
    ].head(20)
)
assert social.loc[social["topic"].eq(-1), "semantic_group"].isna().all()


In [ ]:
# Group IDs are arbitrary after filtering/reclustering. Inspect the group
# summaries above and enter only reviewed group -> theme assignments here.
# Historical mapping, for reference only:
# {0: 'Geopolitical', 3: 'Market Price', 6: 'Developer Activity',
#  9: 'Rental Legal', 12: 'Location'}
REVIEWED_SMDI_MAPPING = {}
if not REVIEWED_SMDI_MAPPING:
    raise ValueError("Review the new semantic groups and fill REVIEWED_SMDI_MAPPING before SMDI export.")
if not set(REVIEWED_SMDI_MAPPING).issubset(set(topic_info["semantic_group"])):
    raise ValueError("The reviewed SMDI mapping contains group IDs absent from this run.")
target_groups = list(REVIEWED_SMDI_MAPPING)
print(social.loc[social['semantic_group'].isin(target_groups), 'semantic_group'].value_counts())


In [ ]:
# ============================================================
# CREATE MASTER POST-LEVEL SMDI DATASET
# ============================================================

# Make a copy of ALL posts
smdi_posts = social.copy()

# Assign semantic group to every post based on its BERTopic topic
smdi_posts['semantic_group'] = smdi_posts['topic'].map(topic_to_group)

# Require reviewed assignments even when this cell is run separately.
if not globals().get("REVIEWED_SMDI_MAPPING"):
    raise ValueError("Fill REVIEWED_SMDI_MAPPING after reviewing this run's groups.")
smdi_mapping = dict(REVIEWED_SMDI_MAPPING)
assert -1 not in topic_to_group
assert not known_automoderator_mask(smdi_posts).any()

# Assign SMDI label ONLY to the five selected groups
# All other groups will remain NaN
smdi_posts['SMDI'] = smdi_posts['semantic_group'].map(smdi_mapping)

# ============================================================
# CHECK
# ============================================================

print("Total posts:", len(smdi_posts))
print("Posts with semantic group:",
      smdi_posts['semantic_group'].notna().sum())

print("\nSMDI distribution:")
print(smdi_posts['SMDI'].value_counts(dropna=False))

# Preview
display(
    smdi_posts[
        ['id', 'date', 'topic', 'semantic_group', 'SMDI']
    ].head(20)
)

In [ ]:
# ============================================================
# 5. CREATE WEEK VARIABLE
# ============================================================

smdi_posts['date'] = pd.to_datetime(
    smdi_posts['date'],
    errors='coerce'
)

smdi_posts = smdi_posts.dropna(subset=['date'])

smdi_posts['week'] = (
    smdi_posts['date']
    .dt.to_period('W')
    .apply(lambda r: r.start_time)
)

display(
    smdi_posts[
        ['date', 'week', 'SMDI']
    ].head(10)
)

display(smdi_posts)

In [ ]:
# ============================================================
# SAVE SMDI POSTS
# ============================================================

# Google Drive folder
smdi_path = "/content/drive/MyDrive/Dubai_Real_Estate_Data/SMDI creation"

# Save the post-level SMDI dataset
smdi_posts.to_csv(
    f"{smdi_path}/smdi_posts.csv",
    index=False
)

print("SMDI posts saved successfully!")
print(f"Location: {smdi_path}/smdi_posts.csv")
print(f"Number of posts saved: {len(smdi_posts):,}")

In [ ]:
# ============================================================
# 6. CALCULATE WEEKLY POST FREQUENCY
# ============================================================

weekly_smdi = (
    smdi_posts
    .groupby(['week', 'SMDI'])
    .size()
    .reset_index(name='post_frequency')
)

display(weekly_smdi.head(20))

In [ ]:
# ============================================================
# 7. CREATE WIDE SMDI DATASET
# ============================================================

weekly_smdi_wide = (
    weekly_smdi
    .pivot(
        index='week',
        columns='SMDI',
        values='post_frequency'
    )
    .fillna(0)
    .reset_index()
)

# Make sure all five columns exist
required_smdi = [
    'Geopolitical',
    'Market Price',
    'Developer Activity',
    'Rental Legal',
    'Location'
]

for col in required_smdi:
    if col not in weekly_smdi_wide.columns:
        weekly_smdi_wide[col] = 0

# Put columns in desired order
weekly_smdi_wide = weekly_smdi_wide[
    ['week'] + required_smdi
]

display(weekly_smdi_wide)